# Customer Churn Prediction & Explainable Machine Learning

**Portfolio project:** Telco customer churn classification  
**Stack:** Python, Pandas, Scikit-learn, XGBoost, SHAP, Matplotlib

## Business objective
Build an end-to-end classification workflow that estimates customer churn risk and translates model output into actionable retention insights.

This notebook demonstrates:
- data-quality checks and exploratory analysis
- leakage-aware train/validation/test splitting
- preprocessing with a reproducible `Pipeline`
- Logistic Regression, Random Forest, and XGBoost comparison
- ROC-AUC, PR-AUC, precision, recall, F1, and confusion matrices
- threshold tuning for the retention use case
- interpretable feature effects and SHAP-based explanation
- a ranked churn-risk list for business action

The dataset contains **7,043 customers and 21 original variables**. The churn rate is approximately **26.5%**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
RANDOM_STATE = 42


## 1. Load and profile the data

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


In [ ]:
profile = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_customer_ids": int(df["customerID"].duplicated().sum()),
    "churn_rate": (df["Churn"] == "Yes").mean(),
    "blank_TotalCharges": int((df["TotalCharges"].astype(str).str.strip() == "").sum())
}
pd.Series(profile)


### Data-quality decision
`TotalCharges` is stored as text because **11 rows are blank**. Those same customers have `tenure = 0`, which is consistent with new customers who have not accumulated charges yet. We convert the field to numeric and set those specific missing values to `0`.


In [ ]:
data = df.copy()

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
mask_new_customer = data["tenure"].eq(0) & data["TotalCharges"].isna()
data.loc[mask_new_customer, "TotalCharges"] = 0

print("Missing TotalCharges after treatment:", data["TotalCharges"].isna().sum())


## 2. Exploratory churn analysis

In [ ]:
churn_summary = (
    data["Churn"]
    .value_counts()
    .rename_axis("Churn")
    .to_frame("Customers")
)
churn_summary["Share"] = churn_summary["Customers"] / churn_summary["Customers"].sum()
display(churn_summary)

ax = churn_summary["Customers"].plot(kind="bar", title="Customer churn distribution")
ax.set_ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
def churn_rate_table(column):
    out = (
        data.groupby(column)["Churn"]
        .apply(lambda s: (s == "Yes").mean())
        .sort_values(ascending=False)
        .rename("Churn Rate")
        .to_frame()
    )
    out["Churn Rate"] = out["Churn Rate"].map(lambda x: f"{x:.1%}")
    return out

for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity"]:
    print(f"\n{col}")
    display(churn_rate_table(col))


### Initial descriptive findings
On this dataset, churn is especially high among:
- month-to-month contracts
- fiber-optic internet customers
- electronic-check customers
- customers without tech support or online security

These are **associations**, not causal effects. The ML section evaluates predictive value jointly across features.


## 3. Train / validation / test split

In [ ]:
y = (data["Churn"] == "Yes").astype(int)
X = data.drop(columns=["Churn", "customerID"])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean().round(4))
print("Validation churn rate:", y_val.mean().round(4))
print("Test churn rate:", y_test.mean().round(4))


## 4. Reproducible preprocessing pipeline

In [ ]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


## 5. Model comparison

In [ ]:
negative_positive_ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2,
        reg_alpha=0.1,
        eval_metric="logloss",
        scale_pos_weight=negative_positive_ratio,
        random_state=RANDOM_STATE,
        n_jobs=4
    )
}

fitted_models = {}
rows = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)

    probability = pipe.predict_proba(X_val)[:, 1]
    prediction = (probability >= 0.50).astype(int)

    rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_val, probability),
        "PR-AUC": average_precision_score(y_val, probability),
        "Precision": precision_score(y_val, prediction),
        "Recall": recall_score(y_val, prediction),
        "F1": f1_score(y_val, prediction),
        "Accuracy": accuracy_score(y_val, prediction)
    })
    fitted_models[name] = pipe

model_results = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False)
display(model_results.style.format({
    "ROC-AUC": "{:.3f}",
    "PR-AUC": "{:.3f}",
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1": "{:.3f}",
    "Accuracy": "{:.3f}"
}))


### Expected benchmark
Using `random_state=42`, the uploaded dataset produced approximately:
- Logistic Regression ROC-AUC: **0.859**
- Random Forest ROC-AUC: **0.856**
- XGBoost ROC-AUC: **0.857**

This is useful as a reproducibility check, not as a performance target to artificially optimize.


In [ ]:
fig, ax = plt.subplots()
for name, pipe in fitted_models.items():
    RocCurveDisplay.from_estimator(pipe, X_val, y_val, name=name, ax=ax)
ax.set_title("Validation ROC curves")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
for name, pipe in fitted_models.items():
    PrecisionRecallDisplay.from_estimator(pipe, X_val, y_val, name=name, ax=ax)
ax.set_title("Validation precision-recall curves")
plt.tight_layout()
plt.show()


## 6. Threshold tuning

In [ ]:
best_model_name = model_results.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

val_probability = best_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_probability)

f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = np.argmax(f1_values)
best_threshold = thresholds[best_idx]

print("Selected model:", best_model_name)
print("F1-optimized validation threshold:", round(float(best_threshold), 3))
print("Validation precision:", round(float(precision[best_idx]), 3))
print("Validation recall:", round(float(recall[best_idx]), 3))
print("Validation F1:", round(float(f1_values[best_idx]), 3))


Threshold selection should follow business costs. A retention team may prefer higher recall if missing a true churner is more expensive than contacting an extra non-churner. Here we demonstrate an F1-optimized threshold, while keeping the business trade-off explicit.


## 7. Final test-set evaluation

In [ ]:
test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= best_threshold).astype(int)

test_metrics = pd.Series({
    "ROC-AUC": roc_auc_score(y_test, test_probability),
    "PR-AUC": average_precision_score(y_test, test_probability),
    "Precision": precision_score(y_test, test_prediction),
    "Recall": recall_score(y_test, test_prediction),
    "F1": f1_score(y_test, test_prediction),
    "Accuracy": accuracy_score(y_test, test_prediction),
    "Threshold": best_threshold
})

display(test_metrics.round(3))

ConfusionMatrixDisplay.from_predictions(y_test, test_prediction)
plt.title("Test confusion matrix")
plt.tight_layout()
plt.show()


With the current split and F1-tuned Logistic Regression threshold, the held-out test set should be around:
- ROC-AUC **0.832**
- Recall **0.759**
- Precision **0.535**
- F1 **0.628**

The exact purpose is not to claim state-of-the-art performance, but to demonstrate a robust, explainable classification workflow.


## 8. Explainability — Logistic Regression feature effects

In [ ]:
# Recover transformed feature names.
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
coefs = best_model.named_steps["model"].coef_[0]

coef_table = (
    pd.DataFrame({"Feature": feature_names, "Coefficient": coefs})
    .assign(Absolute=lambda d: d["Coefficient"].abs())
    .sort_values("Absolute", ascending=False)
)

display(coef_table.head(20))

top_effects = pd.concat([
    coef_table.sort_values("Coefficient").head(10),
    coef_table.sort_values("Coefficient", ascending=False).head(10)
]).sort_values("Coefficient")

ax = top_effects.plot(
    kind="barh",
    x="Feature",
    y="Coefficient",
    legend=False,
    title="Strongest Logistic Regression feature effects"
)
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()


Interpretation:
- Positive coefficients increase predicted churn risk, holding other transformed features constant.
- Negative coefficients reduce predicted churn risk.
- One-hot encoded category coefficients are relative to the model's encoded reference structure.

These are predictive associations, not causal effects.


## 9. Optional SHAP explanation for XGBoost

In [ ]:
# Optional: global SHAP explanation for the fitted XGBoost pipeline.
# If this cell is slow on your machine, use a sample of the validation data.

import shap

xgb_pipe = fitted_models["XGBoost"]
X_val_transformed = xgb_pipe.named_steps["preprocess"].transform(X_val)
feature_names_xgb = xgb_pipe.named_steps["preprocess"].get_feature_names_out()

explainer = shap.TreeExplainer(xgb_pipe.named_steps["model"])
sample_n = min(1000, X_val_transformed.shape[0])
shap_values = explainer.shap_values(X_val_transformed[:sample_n])

shap.summary_plot(
    shap_values,
    X_val_transformed[:sample_n],
    feature_names=feature_names_xgb,
    show=True
)


## 10. Create a business-ready churn-risk list

In [ ]:
# Score the full cleaned dataset with the selected model.
full_X = data.drop(columns=["Churn", "customerID"])
risk_probability = best_model.predict_proba(full_X)[:, 1]

risk_table = data[[
    "customerID", "Contract", "InternetService", "PaymentMethod",
    "tenure", "MonthlyCharges", "TotalCharges", "Churn"
]].copy()

risk_table["Predicted_Churn_Probability"] = risk_probability
risk_table["Risk_Band"] = pd.cut(
    risk_table["Predicted_Churn_Probability"],
    bins=[-np.inf, 0.30, 0.60, np.inf],
    labels=["Low", "Medium", "High"]
)

risk_table = risk_table.sort_values(
    "Predicted_Churn_Probability", ascending=False
)

display(risk_table.head(20))
risk_table.to_csv("customer_churn_risk_scores.csv", index=False)


## 11. Business interpretation and limitations

### Potential retention use
A retention team could prioritize customers with high predicted churn probability and combine the score with customer value, tenure, and contact cost.

### Key limitations
- The dataset is a static snapshot, so true temporal churn forecasting is not possible.
- The project demonstrates predictive association, not causal impact.
- No intervention/campaign outcomes are available, so retention ROI cannot be directly estimated.
- Threshold choice should be driven by business costs, capacity, and the value of retained customers.
- Model fairness and subgroup performance should be audited before operational use.

### Recommended production extensions
- probability calibration
- drift monitoring
- subgroup performance checks
- cost-sensitive threshold optimization
- scheduled model retraining
- experiment design to evaluate retention interventions
